# Split Wav2Vec2 (stability) embeddings by utterance type (C/O/W)

This notebook reads the existing Wav2Vec2 stability embeddings and splits them into 3 datasets based on the filename suffix:

- `*_C<number>.wav` → **Closed** (C)
- `*_O<number>.wav` → **Open** (O)
- `*_W<number>.wav` → **Words** (W)

It saves each subset under `embeddings/wav2vec/utterance_types/...` with the same file naming (`xwav2vec2_stability.npy`, `labels.npy`, `subject_ids.npy`, `file_list.npy`, `label_map.npy`).


In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np

# ===================== Paths (edit if needed) =====================
BASE = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta")
SRC_DIR = BASE / "embeddings" / "wav2vec"
X_PATH = SRC_DIR / "xwav2vec2_stability.npy"  # best model+config you selected

OUT_ROOT = SRC_DIR / "utterance_types"
OUT_DIRS = {
    "C": OUT_ROOT / "closed_C",
    "O": OUT_ROOT / "open_O",
    "W": OUT_ROOT / "words_W",
}

print("SRC_DIR:", SRC_DIR, "| exists:", SRC_DIR.is_dir())
print("X_PATH :", X_PATH, "| exists:", X_PATH.is_file())
for k, d in OUT_DIRS.items():
    print(f"OUT[{k}]:", d)


SRC_DIR: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec | exists: True
X_PATH : /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/xwav2vec2_stability.npy | exists: True
OUT[C]: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/closed_C
OUT[O]: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/open_O
OUT[W]: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/words_W


In [2]:
# ===================== Load (must be aligned row-wise) =====================
X = np.load(X_PATH)
y = np.load(SRC_DIR / "labels.npy", allow_pickle=True)
subject_ids = np.load(SRC_DIR / "subject_ids.npy", allow_pickle=True).astype(str)
file_list = np.load(SRC_DIR / "file_list.npy", allow_pickle=True).astype(str)
label_map = np.load(SRC_DIR / "label_map.npy", allow_pickle=True)

assert len(X) == len(y) == len(subject_ids) == len(file_list), "Length mismatch between X/y/subject_ids/file_list"
print("Loaded:")
print("  X         :", X.shape)
print("  y         :", y.shape)
print("  subject_ids:", subject_ids.shape)
print("  file_list :", file_list.shape)
print("  label_map :", label_map)


Loaded:
  X         : (6926, 3072)
  y         : (6926,)
  subject_ids: (6926,)
  file_list : (6926,)
  label_map : {'Control': 0, 'ALSwDysarthria': 1, 'ALSwoDysarthria': 2}


In [3]:
# ===================== Split by utterance type =====================
# Example: Control/F1/..._C12.wav  → type='C'
pat = re.compile(r"_(C|O|W)(\d+)\.wav$", re.IGNORECASE)

utt_type = np.empty(len(file_list), dtype=object)
for i, fp in enumerate(file_list):
    m = pat.search(fp)
    if not m:
        raise ValueError(f"Could not parse utterance type from: {fp}")
    utt_type[i] = m.group(1).upper()

counts = {k: int(np.sum(utt_type == k)) for k in ["C", "O", "W"]}
print("Utterance-type counts:", counts)

# ===================== Save subsets =====================
for k, out_dir in OUT_DIRS.items():
    idx = np.where(utt_type == k)[0]
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / "xwav2vec2_stability.npy", X[idx])
    np.save(out_dir / "labels.npy", y[idx], allow_pickle=True)
    np.save(out_dir / "subject_ids.npy", subject_ids[idx], allow_pickle=True)
    np.save(out_dir / "file_list.npy", file_list[idx], allow_pickle=True)
    np.save(out_dir / "label_map.npy", label_map, allow_pickle=True)

    print(f"\nSaved [{k}] -> {out_dir}")
    print("  X:", (out_dir / "xwav2vec2_stability.npy"), "shape=", X[idx].shape)
    print("  y:", (out_dir / "labels.npy"), "n=", len(idx))
    print("  subjects unique=", len(np.unique(subject_ids[idx])))

print("\nDone.")


Utterance-type counts: {'C': 2462, 'O': 1138, 'W': 3326}

Saved [C] -> /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/closed_C
  X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/closed_C/xwav2vec2_stability.npy shape= (2462, 3072)
  y: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/closed_C/labels.npy n= 2462
  subjects unique= 63

Saved [O] -> /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/open_O
  X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/open_O/xwav2vec2_stability.npy shape= (1138, 3072)
  y: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/open_O/labels.npy n= 1138
  subjects unique= 63

Saved [W] -> /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/words_W
  X: /mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/wav2vec/utterance_types/words_W/xwav2vec2_stabilit